In [3]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")

/home/dl01e26/.conda/envs/vla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...


In [4]:
for n, p in policy.named_parameters():   # inspecting the insane structure behind the model
    print(n, p.shape, p.requires_grad) 

model.vlm_with_expert.vlm.model.vision_model.embeddings.patch_embedding.weight torch.Size([768, 3, 16, 16]) False
model.vlm_with_expert.vlm.model.vision_model.embeddings.patch_embedding.bias torch.Size([768]) False
model.vlm_with_expert.vlm.model.vision_model.embeddings.position_embedding.weight torch.Size([1024, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.k_proj.weight torch.Size([768, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.k_proj.bias torch.Size([768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.v_proj.weight torch.Size([768, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.v_proj.bias torch.Size([768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.q_proj.weight torch.Size([768, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.q_proj.bias torch.Size([768]) False
mode

In [5]:
print("total params:", sum(p.numel() for p in policy.parameters()) / 1000000, "mil")
print("trainable params:", sum(p.numel() for p in policy.parameters() if p.requires_grad) / 1000000, "mil")

total params: 450.046176 mil
trainable params: 99.880992 mil


In [6]:
#setting up dataset and dataloader
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from torch.utils.data import DataLoader
import torch

dataset = LeRobotDataset(
    repo_id= "local/pickcube",
    root="/home/dl01e26/vla_pipeline/lerobot_demos"
)

sample = dataset[0]

print(sample["action"].shape) #random sample of the data that should give 8

print(dataset.meta.features.keys())   # or dataset.features
print(dataset.meta.camera_keys)
print(dataset.meta.stats.keys())


torch.Size([7])
dict_keys(['action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'task', 'observation.images.base_camera'])
['observation.images.base_camera']
dict_keys(['action', 'observation.state', 'observation.images.base_camera', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'])


In [7]:
#The base dataset has a mismatch between the expected feature names the actual feature names
# so we tell the model what features to expect
from lerobot.configs.types import FeatureType
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

#Finding the features and the config for the model
features = dataset_to_policy_features(dataset.meta.features)
cfg = PreTrainedConfig.from_pretrained("lerobot/smolvla_base")
cfg.output_features = {k: f for k, f in features.items() if f.type is FeatureType.ACTION}
cfg.input_features = {k: f for k, f in features.items() if k not in cfg.output_features}

#initializing the model with the right config for the dataset
policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base", config = cfg)

#we make our pre and post processor because we need tokens, not text as well as normalization.
preprocess, postprocess = make_pre_post_processors(
    policy.config, pretrained_path="lerobot/smolvla_base",
    dataset_stats=dataset.meta.stats,
)

#we remake the dataset with delta_timestamps
from lerobot.datasets.factory import resolve_delta_timestamps

delta_timestamps = resolve_delta_timestamps(policy.config, dataset.meta)
dataset = LeRobotDataset(
    "local/pickcube",
    root = "/home/dl01e26/vla_pipeline/lerobot_demos",
    delta_timestamps = delta_timestamps,
)

data_loader = DataLoader(dataset)

#to see if it works
batch = preprocess(next(iter(data_loader)))
with torch.no_grad():
    loss, info = policy.forward(batch)
    print(loss)
    print(info)

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...
tensor(0.2158, device='cuda:0')
{'losses_after_forward': 0.21576616168022156, 'losses_after_rm_padding': 0.21576616168022156, 'loss': 0.21576616168022156}


In [ ]:
lr = 1e-4
weight_decay = 0.01
nEpoch = 2

params_to_update = []
for name, param in policy.named_parameters():
    if param.requires_grad:
        params_to_update.append(param)

opt = torch.optim.AdamW(params_to_update, lr = lr, weight_decay = weight_decay)
# working on being able to change the loss function

train_loss = []

policy.train()

print("BEGINNING TRAINING \n ----------------------------------------------------------------------")
for iEpoch in range(nEpoch):
    tot_loss = 0
    for batch in data_loader:
        opt.zero_grad()

        batch = preprocess(batch)
        
        loss, info = policy.forward(batch)

        loss.backward()

        opt.step()

        tot_loss += loss.item()

    train_loss.append(loss.item())

    print(f"Epoch {iEpoch} had loss {tot_loss/len(dataset)}")
    

BEGINNING TRAINING 
 ----------------------------------------------------------------------


In [ ]:
import gymnasium as gym
import mani_skill.envs

env = gym.make(
    "PushCube-v1", # there are more tasks e.g. "PickCube-v1", "PegInsertionSide-v1", ...
    num_envs=1,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_ee_delta_pose", # there is also "pd_joint_delta_pos", ...
    render_mode = "rgb_array"
)
print("Observation space", env.observation_space)
print("Action space", env.action_space)

obs, _ = env.reset(seed=0) # reset with a seed for determinism
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    env.render()  # a display is required to render
env.close()

Observation space Box(-inf, inf, (1, 35), float32)
Action space Box(-1.0, 1.0, (7,), float32)


In [ ]:
#Evaluation copied from the maniskill docs
import gymnasium as gym
import torch
from collections import defaultdict
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv
env_id = "PushCube-v1"
num_eval_envs = 64
env_kwargs = dict(obs_mode="state") # modify your env_kwargs here
eval_envs = gym.make(env_id, num_envs=num_eval_envs, reconfiguration_freq=1, **env_kwargs)
# add any other wrappers here
eval_envs = ManiSkillVectorEnv(eval_envs, ignore_terminations=True, record_metrics=True)

# evaluation loop, which will record metrics for complete episodes only
obs, _ = eval_envs.reset(seed=0)
eval_metrics = defaultdict(list)
for _ in range(400):
    action = eval_envs.action_space.sample() # replace with your policy action
    obs, rew, terminated, truncated, info = eval_envs.step(action)
    # note as there are no partial resets, truncated is True for all environments at the same time
    if truncated.any():
        for k, v in info["final_info"]["episode"].items():
            eval_metrics[k].append(v.float())
for k in eval_metrics.keys():
    print(f"{k}_mean: {torch.mean(torch.stack(eval_metrics[k])).item()}")
    